# Test de l'agent de preprocessing sur differents datasets

Ce notebook teste l'agent sur plusieurs jeux de donnees aux caracteristiques variees.

**Pipeline actuel :** 12 noeuds, 3 points d'interruption (domain, transformations, outliers) + alerte leakage optionnelle.

**Datasets testes :**
1. **Titanic** - Classique, valeurs manquantes, mix num/cat
2. **Tips** - Petit dataset, peu de features, simple
3. **Diamonds** - Grand dataset (54k lignes), haute cardinalite
4. **Penguins** - Petit dataset avec NaN, colonnes categoriques
5. **Dataset synthetique** - Cas extremes fabriques (colonnes constantes, 100% NaN, etc.)
6. **Target invalide** - Colonne target inexistante

In [16]:
import json
import time
import pandas as pd
import numpy as np
from pathlib import Path

from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command

from src.agents.pipeline import build_pipeline

In [17]:
def run_pipeline_auto(df_path: str, target: str, thread_id: str = "test"):
    """Lance le pipeline complet avec auto-approve a chaque interrupt."""

    checkpointer = MemorySaver()
    pipeline = build_pipeline(checkpointer=checkpointer)
    config = {"configurable": {"thread_id": thread_id}}

    results = {"df_path": df_path, "target": target, "errors": []}
    start = time.time()

    try:
        # Etape 1 : Analyse -> interrupt domain_review
        pipeline.invoke({"df_path": df_path, "target": target}, config=config)

        snapshot = pipeline.get_state(config)
        domain_interrupt = snapshot.tasks[0].interrupts[0].value
        results["domain_context"] = domain_interrupt.get("domain_context", {})
        results["semantic_anomalies"] = domain_interrupt.get("semantic_anomalies", [])

        # Auto-approve domain
        pipeline.invoke(Command(resume="approve"), config=config)

        # Etape 2 : Propositions de transformations -> interrupt transformation_review
        snapshot = pipeline.get_state(config)
        transform_interrupt = snapshot.tasks[0].interrupts[0].value
        results["transform_proposals"] = transform_interrupt.get("proposals", {})
        results["leakage_alerts"] = transform_interrupt.get("leakage_alerts", [])
        results["correlation_alerts"] = transform_interrupt.get("correlation_alerts", [])

        # Auto-approve transformations
        pipeline.invoke(Command(resume="approve"), config=config)

        # Etape 2b : Handle optional leakage_warning interrupt
        snapshot = pipeline.get_state(config)
        if snapshot.tasks and snapshot.tasks[0].interrupts:
            interrupt_value = snapshot.tasks[0].interrupts[0].value
            if interrupt_value.get("type") == "leakage_warning":
                results["leakage_warning"] = True
                pipeline.invoke(Command(resume="approve"), config=config)
                snapshot = pipeline.get_state(config)

        # Etape 3 : Propositions outliers -> interrupt outlier_review
        if snapshot.tasks and snapshot.tasks[0].interrupts:
            outlier_interrupt = snapshot.tasks[0].interrupts[0].value
            results["outlier_proposals"] = outlier_interrupt.get("proposals", {})

            # Auto-approve outliers
            state = pipeline.invoke(Command(resume="approve"), config=config)
        else:
            state = snapshot.values

        results["report"] = state.get("report", {})
        results["quality_score"] = state.get("quality_score", {})
        results["confidence_map"] = state.get("confidence_map", [])
        results["final_state"] = state
        results["success"] = True

    except Exception as e:
        results["success"] = False
        results["errors"].append(f"{type(e).__name__}: {e}")

    results["duration"] = round(time.time() - start, 1)
    return results


def print_summary(results: dict):
    """Affiche un resume compact des resultats."""
    status = "OK" if results["success"] else "ECHEC"
    print(f"\nDataset: {results['df_path']}")
    print(f"Target: {results['target']}")
    print(f"Statut: {status} | Duree: {results['duration']}s")

    if not results["success"]:
        for err in results["errors"]:
            print(f"  ERREUR: {err}")
        return

    # Domain
    domain = results.get("domain_context", {})
    if domain:
        print(f"\nDomaine infere: {domain.get('domain', '?')} - {domain.get('description', '')}")

    # Transformations
    transforms = results.get("transform_proposals", {}).get("transformations", [])
    rule_count = sum(1 for t in transforms if t.get("source") == "rule")
    llm_count = len(transforms) - rule_count
    print(f"\nTransformations: {len(transforms)} ({rule_count} rule, {llm_count} LLM)")
    for t in transforms:
        src = t.get("source", "llm")
        print(f"  [{src}] {t['column']}: {t['action']}")

    # Leakage
    leakage = results.get("leakage_alerts", [])
    if leakage:
        print(f"\nLeakage alerts: {len(leakage)}")
        for l in leakage:
            print(f"  {l['column']}: corr={l['correlation']}")

    # Correlations
    correlations = results.get("correlation_alerts", [])
    if correlations:
        print(f"\nCorrelation alerts: {len(correlations)} pair(s)")
        for c in correlations:
            method = "Pearson" if c["method"] == "pearson" else "Cramér's V"
            print(f"  {c['column_a']} <-> {c['column_b']}: {c['correlation']:.3f} ({method}, {c['severity']})")
            for s in c.get("suggested_actions", []):
                print(f"    -> {s}")

    # Outliers
    outliers = results.get("outlier_proposals", {}).get("outlier_actions", [])
    print(f"\nOutliers: {len(outliers)} actions")
    for o in outliers:
        print(f"  [{o.get('source', 'llm')}] {o['column']}: {o['method']}/{o['action']}")

    # Quality score
    qs = results.get("quality_score", {})
    if qs:
        print(f"\nQuality score: {qs.get('overall', '?')}/100")
        print(f"  Completeness: {qs.get('completeness', '?')}/30")
        print(f"  Type consistency: {qs.get('type_consistency', '?')}/15")
        print(f"  Leakage risk: {qs.get('leakage_risk', '?')}")
        print(f"  Outlier coverage: {qs.get('outlier_coverage', '?')}/20")
        print(f"  Drop penalty: {qs.get('drop_penalty', 0)}")

    # Confidence
    conf_map = results.get("confidence_map", [])
    if conf_map:
        total = len(conf_map)
        rule_ct = sum(1 for c in conf_map if c.get("source") == "rule")
        print(f"\nConfidence: {total} decisions, {rule_ct}/{total} deterministic ({rule_ct/total*100:.0f}%)")

    # Shape
    report = results.get("report", {})
    ba = report.get("before_after", {})
    if ba:
        print(f"\nShape: {ba.get('original_shape', '?')} -> {ba.get('final_shape', '?')}")

## Test 1 : Titanic (baseline)

Dataset classique avec valeurs manquantes, colonnes categoriques et numeriques.
L'agent a deja ete teste dessus -- sert de reference.

In [18]:
url_titanic = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df_titanic = pd.read_csv(url_titanic)
print(f"Shape: {df_titanic.shape}")
print(f"Types:\n{df_titanic.dtypes.value_counts()}")
print(f"\nValeurs manquantes:\n{df_titanic.isnull().sum()[df_titanic.isnull().sum() > 0]}")
df_titanic.head(3)

Shape: (891, 15)
Types:
str        7
int64      4
float64    2
bool       2
Name: count, dtype: int64

Valeurs manquantes:
age            177
embarked         2
deck           688
embark_town      2
dtype: int64


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True


In [19]:
results_titanic = run_pipeline_auto(url_titanic, "survived", thread_id="titanic")
print_summary(results_titanic)

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


17:46:27 - src.tracking.wandb_tracker - INFO - W&B initialized: project=preprocessing-pipeline, run=titanic
17:46:27 - src.agents.base_agent - INFO - Dataset loaded: 891 rows x 15 columns, target='survived'
17:46:27 - src.agents.base_agent - INFO - Target 'survived': task_type=classification, 2 unique, 0.0% NaN
17:46:27 - src.agents.base_agent - INFO - Dtype audit: 3 columns mistyped
17:46:27 - src.agents.base_agent - INFO - Inferring domain context and anomalies (1 LLM call)
17:46:39 - src.agents.base_agent - INFO - Generating diagnosis (1 LLM call)
17:46:57 - src.agents.base_agent - INFO - Dataset hash (MD5): 6a7a2ed01936ef56e903ba2bb2f6fbc6
17:46:57 - src.agents.base_agent - INFO - Split: 712 train, 179 test (20%) -> data\outputs\titanic_train.csv, data\outputs\titanic_test.csv
17:46:57 - src.agents.pipeline - INFO - Human decision on domain context: approve
17:46:57 - src.agents.transformation_agent - WARNING - Missing pattern: 'embarked' and 'embark_town' have correlated missingne

baseline/baseline_score,▁
baseline/delta,▁
baseline/model_score,▁
report/completeness,▁
report/deterministic_ratio,▁
report/outlier_coverage,▁
report/quality_overall,▁
report/type_consistency,▁
baseline/baseline_score,0.6145
baseline/delta,0.2123
baseline/model_score,0.8268


17:48:16 - src.tracking.wandb_tracker - INFO - W&B run finished



Dataset: https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv
Target: survived
Statut: OK | Duree: 129.0s

Domaine infere: transport - Ce dataset concerne les passagers du Titanic, incluant des informations sur leur survie, classe, sexe, âge, et d'autres caractéristiques. Il est utilisé pour prédire la survie des passagers en fonction de ces attributs.

Transformations: 19 (7 rule, 12 LLM)
  [rule] deck: drop_column
  [llm] age: impute_median
  [llm] deck: drop_column
  [rule] pclass: ordinal_encoding
  [rule] sibsp: ordinal_encoding
  [rule] parch: ordinal_encoding
  [rule] adult_male: ordinal_encoding
  [rule] alone: ordinal_encoding
  [llm] sibsp: ordinal_encoding
  [llm] parch: ordinal_encoding
  [llm] pclass: ordinal_encoding
  [llm] sex: one_hot_encoding
  [llm] embarked: one_hot_encoding
  [llm] class: one_hot_encoding
  [llm] who: one_hot_encoding
  [llm] alive: one_hot_encoding
  [llm] embark_town: one_hot_encoding
  [rule] fare: robust_scaling
  [llm] fa

## Test 2 : Tips (dataset simple)

Petit dataset (244 lignes), peu de features, pas de valeurs manquantes.
Teste si l'agent gere bien un dataset deja "propre".

In [20]:
url_tips = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
df_tips = pd.read_csv(url_tips)
print(f"Shape: {df_tips.shape}")
print(f"Types:\n{df_tips.dtypes.value_counts()}")
print(f"\nValeurs manquantes: {df_tips.isnull().sum().sum()}")
df_tips.head(3)

Shape: (244, 7)
Types:
str        4
float64    2
int64      1
Name: count, dtype: int64

Valeurs manquantes: 0


,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3


In [21]:
results_tips = run_pipeline_auto(url_tips, "tip", thread_id="tips")
print_summary(results_tips)

17:48:19 - src.tracking.wandb_tracker - INFO - W&B initialized: project=preprocessing-pipeline, run=tips
17:48:19 - src.agents.base_agent - INFO - Dataset loaded: 244 rows x 7 columns, target='tip'
17:48:19 - src.agents.base_agent - INFO - Target 'tip': task_type=regression, 123 unique, 0.0% NaN
17:48:19 - src.agents.base_agent - INFO - Dtype audit: 1 columns mistyped
17:48:19 - src.agents.base_agent - INFO - Inferring domain context and anomalies (1 LLM call)
17:48:27 - src.agents.base_agent - INFO - Generating diagnosis (1 LLM call)


17:48:39 - src.agents.base_agent - INFO - Dataset hash (MD5): 21a1749bc760e771c7a011b17e17d17b
17:48:39 - src.agents.base_agent - INFO - Split: 195 train, 49 test (20%) -> data\outputs\tips_train.csv, data\outputs\tips_test.csv
17:48:39 - src.agents.pipeline - INFO - Human decision on domain context: approve
17:48:39 - src.agents.transformation_agent - INFO - Rule-based: 2 transforms, 4 ambiguous
17:48:39 - src.agents.transformation_agent - INFO - LLM for 4 ambiguous columns
17:48:46 - src.agents.transformation_agent - INFO - Inter-feature correlations: 1 pair(s) above 0.75 [0 numeric, 1 categorical]
17:48:46 - src.agents.transformation_agent - INFO - LLM for feature engineering proposals
17:48:53 - src.agents.transformation_agent - INFO - Proposals: 0 auto-drop, 0 dtype, 0 sentinel, 2 rule, 7 LLM = 9 total
17:48:53 - src.agents.pipeline - INFO - Multicollinearity: 1 pair(s) detected (1 very_high, 0 high)
17:48:53 - src.agents.pipeline - INFO -   day <-> time: 0.932 (Cramér's V, very_h

baseline/baseline_score,▁
baseline/delta,▁
baseline/model_score,▁
report/completeness,▁
report/deterministic_ratio,▁
report/outlier_coverage,▁
report/quality_overall,▁
report/type_consistency,▁
baseline/baseline_score,-0.159
baseline/delta,0.5673
baseline/model_score,0.4084


17:50:04 - src.tracking.wandb_tracker - INFO - W&B run finished



Dataset: https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv
Target: tip
Statut: OK | Duree: 108.0s

Domaine infere: restaurant - Ce dataset contient des informations sur les factures et pourboires dans un restaurant, incluant des détails sur les clients tels que leur sexe, s'ils sont fumeurs, le jour de la semaine, le moment du repas et la taille de la table.

Transformations: 9 (2 rule, 7 LLM)
  [llm] total_bill: drop_rows
  [llm] tip: drop_rows
  [rule] size: ordinal_encoding
  [llm] size: ordinal_encoding
  [llm] sex: label_encoding
  [llm] smoker: label_encoding
  [llm] day: one_hot_encoding
  [llm] time: label_encoding
  [rule] total_bill: robust_scaling

Correlation alerts: 1 pair(s)
  day <-> time: 0.932 (Cramér's V, very_high)
    -> drop_least_informative: supprimer la variable avec le moins de variance ou la moins corrélée au target
    -> pca: réduction de dimension sur le groupe de variables corrélées (>= 3)
    -> keep: certains modèles (arbres, gradie

## Test 3 : Diamonds (grand dataset, haute cardinalite)

54k lignes, colonne `cut` ordinale, `color`/`clarity` categoriques a haute cardinalite.
Teste les performances sur un gros volume et la gestion du one-hot encoding sur des colonnes a beaucoup de modalites.

In [22]:
url_diamonds = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv"
df_diamonds = pd.read_csv(url_diamonds)
print(f"Shape: {df_diamonds.shape}")
print(f"Types:\n{df_diamonds.dtypes.value_counts()}")
print(f"\nCardinalite des categoriques:")
for col in df_diamonds.select_dtypes(include="object").columns:
    print(f"  {col}: {df_diamonds[col].nunique()} valeurs uniques")
df_diamonds.head(3)

Shape: (53940, 10)
Types:
float64    6
str        3
int64      1
Name: count, dtype: int64

Cardinalite des categoriques:
  cut: 5 valeurs uniques
  color: 7 valeurs uniques
  clarity: 8 valeurs uniques


C:\Users\abdel\AppData\Local\Temp\ipykernel_18560\3384341890.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_diamonds.select_dtypes(include="object").columns:


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31


In [23]:
results_diamonds = run_pipeline_auto(url_diamonds, "price", thread_id="diamonds")
print_summary(results_diamonds)

17:50:08 - src.tracking.wandb_tracker - INFO - W&B initialized: project=preprocessing-pipeline, run=diamonds
17:50:09 - src.agents.base_agent - INFO - Dataset loaded: 53940 rows x 10 columns, target='price'
17:50:09 - src.agents.base_agent - INFO - Target 'price': task_type=regression, 11602 unique, 0.0% NaN
17:50:09 - src.agents.base_agent - INFO - Inferring domain context and anomalies (1 LLM call)
17:50:19 - src.agents.base_agent - INFO - Generating diagnosis (1 LLM call)
17:50:32 - src.agents.base_agent - INFO - Dataset hash (MD5): a154da86d0a81976bc0fb5ccbc40e256
17:50:33 - src.agents.base_agent - INFO - Split: 43152 train, 10788 test (20%) -> data\outputs\diamonds_train.csv, data\outputs\diamonds_test.csv
17:50:33 - src.agents.pipeline - INFO - Human decision on domain context: approve
17:50:33 - src.agents.transformation_agent - INFO - Rule-based: 6 transforms, 3 ambiguous
17:50:33 - src.agents.transformation_agent - INFO - LLM for 3 ambiguous columns
17:50:46 - src.agents.trans

baseline/baseline_score,▁
baseline/delta,▁
baseline/model_score,▁
report/completeness,▁
report/deterministic_ratio,▁
report/outlier_coverage,▁
report/quality_overall,▁
report/type_consistency,▁
baseline/baseline_score,-0.0001
baseline/delta,0.8727
baseline/model_score,0.8727


17:51:57 - src.tracking.wandb_tracker - INFO - W&B run finished



Dataset: https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv
Target: price
Statut: OK | Duree: 110.8s

Domaine infere: e-commerce - Ce dataset contient des informations sur des diamants, incluant leurs caractéristiques physiques et leur prix. Il est utilisé pour prédire le prix des diamants en fonction de leurs attributs.

Transformations: 23 (6 rule, 17 LLM)
  [llm] carat: drop_rows
  [llm] depth: drop_rows
  [llm] table: drop_rows
  [llm] x: drop_rows
  [llm] y: drop_rows
  [llm] z: drop_rows
  [llm] cut: one_hot_encoding
  [llm] color: one_hot_encoding
  [llm] clarity: one_hot_encoding
  [rule] carat: robust_scaling
  [rule] depth: standard_scaling
  [rule] table: standard_scaling
  [rule] x: standard_scaling
  [rule] y: log_transform
  [rule] z: log_transform
  [llm] y: log_transform
  [llm] z: log_transform
  [llm] carat: standard_scaling
  [llm] depth: standard_scaling
  [llm] table: standard_scaling
  [llm] x: standard_scaling
  [llm] y: standard_scaling


## Test 4 : Penguins (petit dataset avec NaN naturels)

344 lignes, 3 especes, valeurs manquantes dans plusieurs colonnes numeriques et categoriques.
Teste la gestion des NaN disperses sur peu de donnees.

In [24]:
url_penguins = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
df_penguins = pd.read_csv(url_penguins)
print(f"Shape: {df_penguins.shape}")
print(f"Types:\n{df_penguins.dtypes.value_counts()}")
print(f"\nValeurs manquantes:\n{df_penguins.isnull().sum()[df_penguins.isnull().sum() > 0]}")
df_penguins.head(3)

Shape: (344, 7)
Types:
float64    4
str        3
Name: count, dtype: int64

Valeurs manquantes:
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE


In [25]:
results_penguins = run_pipeline_auto(url_penguins, "species", thread_id="penguins")
print_summary(results_penguins)

17:52:00 - src.tracking.wandb_tracker - INFO - W&B initialized: project=preprocessing-pipeline, run=penguins
17:52:00 - src.agents.base_agent - INFO - Dataset loaded: 344 rows x 7 columns, target='species'
17:52:00 - src.agents.base_agent - INFO - Target 'species': task_type=classification, 3 unique, 0.0% NaN
17:52:00 - src.agents.base_agent - INFO - Inferring domain context and anomalies (1 LLM call)
17:52:08 - src.handlers.error_handler - WARNING - Validation failed (attempt 1/3): - Champ 'anomalies -> 0 -> sentinel_values -> 0 -> float': Input should be a valid number (valeur recue: None)
- Champ 'anomalies -> 0 -> sentinel_values -> 0 -> int': Input should be a valid integer (valeur recue: None)
- Champ 'anomalies -> 0 -> sentinel_values -> 0 -> str': Input should be a valid string (valeur recue: None)
- Champ 'anomalies -> 1 -> sentinel_values -> 0 -> float': Input should be a valid number (valeur recue: None)
- Champ 'anomalies -> 1 -> sentinel_values -> 0 -> int': Input should b


Dataset: https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv
Target: species
Statut: ECHEC | Duree: 32.7s
  ERREUR: ValueError: LLM response failed validation after 3 attempts. Last error: 15 validation errors for DomainAndAnomaliesResponse
anomalies.0.sentinel_values.0.float
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.12/v/float_type
anomalies.0.sentinel_values.0.int
  Input should be a valid integer [type=int_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.12/v/int_type
anomalies.0.sentinel_values.0.str
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type
anomalies.1.sentinel_values.0.float
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneTy

## Test 5 : Dataset synthetique (cas extremes)

Dataset fabrique pour pousser l'agent dans ses limites :
- Colonne 100% NaN
- Colonne constante (variance 0)
- Colonne categorique a tres haute cardinalite (IDs uniques)
- Colonne avec des valeurs extremes
- Target binaire
- Colonne datetime (non supportee actuellement)

In [26]:
np.random.seed(42)
n = 500

df_extreme = pd.DataFrame({
    "target": np.random.choice([0, 1], size=n),
    "normal_col": np.random.randn(n),
    "all_nan_col": [np.nan] * n,
    "constant_col": [42] * n,
    "high_cardinality_id": [f"ID_{i}" for i in range(n)],
    "extreme_values": np.concatenate([np.random.randn(n - 5), [1e6, -1e6, 1e7, -1e7, 1e8]]),
    "binary_cat": np.random.choice(["yes", "no"], size=n),
    "datetime_col": pd.date_range("2020-01-01", periods=n, freq="h"),
    "mostly_nan_col": np.where(np.random.rand(n) < 0.95, np.nan, np.random.randn(n)),
})

# Sauvegarder localement
extreme_path = "data/outputs/extreme_test.csv"
Path(extreme_path).parent.mkdir(parents=True, exist_ok=True)
df_extreme.to_csv(extreme_path, index=False)

print(f"Shape: {df_extreme.shape}")
print(f"\nTypes:\n{df_extreme.dtypes}")
print(f"\nValeurs manquantes:\n{df_extreme.isnull().sum()}")
print(f"\nCardinalite:")
for col in df_extreme.select_dtypes(include="object").columns:
    print(f"  {col}: {df_extreme[col].nunique()}")
df_extreme.head(3)

Shape: (500, 9)

Types:
target                          int64
normal_col                    float64
all_nan_col                   float64
constant_col                    int64
high_cardinality_id               str
extreme_values                float64
binary_cat                        str
datetime_col           datetime64[us]
mostly_nan_col                float64
dtype: object

Valeurs manquantes:
target                   0
normal_col               0
all_nan_col            500
constant_col             0
high_cardinality_id      0
extreme_values           0
binary_cat               0
datetime_col             0
mostly_nan_col         477
dtype: int64

Cardinalite:
  high_cardinality_id: 500
  binary_cat: 2


C:\Users\abdel\AppData\Local\Temp\ipykernel_18560\3856593254.py:25: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_extreme.select_dtypes(include="object").columns:


,target,normal_col,all_nan_col,constant_col,high_cardinality_id,extreme_values,binary_cat,datetime_col,mostly_nan_col
0,0,-0.846794,NaN,42,ID_0,0.710960,no,2020-01-01 00:00:00,NaN
1,1,-1.514847,NaN,42,ID_1,0.444263,no,2020-01-01 01:00:00,NaN
2,0,-0.446515,NaN,42,ID_2,-0.360966,yes,2020-01-01 02:00:00,NaN


In [27]:
results_extreme = run_pipeline_auto(extreme_path, "target", thread_id="extreme")
print_summary(results_extreme)

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


17:52:39 - src.tracking.wandb_tracker - INFO - W&B initialized: project=preprocessing-pipeline, run=extreme_test
17:52:39 - src.agents.base_agent - INFO - Dataset loaded: 500 rows x 9 columns, target='target'
17:52:39 - src.agents.base_agent - INFO - Target 'target': task_type=classification, 2 unique, 0.0% NaN
17:52:39 - src.agents.base_agent - INFO - Dtype audit: 2 columns mistyped
17:52:39 - src.agents.base_agent - INFO - Auto-drop: ['all_nan_col', 'constant_col']
17:52:40 - src.agents.base_agent - INFO - Inferring domain context and anomalies (1 LLM call)
17:52:46 - src.agents.base_agent - INFO - Generating diagnosis (1 LLM call)
17:52:59 - src.agents.base_agent - INFO - Dataset hash (MD5): 83b4a3816dfbed683f731ff038ccfde8
17:53:00 - src.agents.base_agent - INFO - Split: 400 train, 100 test (20%) -> data\outputs\extreme_test_train.csv, data\outputs\extreme_test_test.csv
17:53:00 - src.agents.pipeline - INFO - Human decision on domain context: approve
17:53:00 - src.agents.transform

baseline/baseline_score,▁
baseline/delta,▁
baseline/model_score,▁
report/completeness,▁
report/deterministic_ratio,▁
report/outlier_coverage,▁
report/quality_overall,▁
report/type_consistency,▁
baseline/baseline_score,0.51
baseline/delta,0.01
baseline/model_score,0.52


17:54:02 - src.tracking.wandb_tracker - INFO - W&B run finished



Dataset: data/outputs/extreme_test.csv
Target: target
Statut: OK | Duree: 89.7s

Domaine infere: autre - Ce dataset semble être utilisé pour une tâche de classification, probablement dans un contexte d'analyse de données où l'on cherche à prédire une variable cible à partir de diverses caractéristiques. Les colonnes incluent des valeurs numériques, des identifiants à haute cardinalité, des colonnes avec des valeurs manquantes et des colonnes contenant des valeurs extrêmes.

Transformations: 11 (5 rule, 6 LLM)
  [rule] all_nan_col: drop_column
  [rule] constant_col: drop_column
  [rule] mostly_nan_col: drop_column
  [llm] mostly_nan_col: drop_column
  [llm] high_cardinality_id: one_hot_encoding
  [llm] binary_cat: label_encoding
  [llm] datetime_col: extract_datetime
  [rule] normal_col: standard_scaling
  [rule] extreme_values: standard_scaling
  [llm] extreme_values: log_transform
  [llm] normal_col: standard_scaling

Correlation alerts: 3 pair(s)
  high_cardinality_id <-> binary_cat

## Test 6 : Target invalide

Que se passe-t-il si on donne une colonne target qui n'existe pas dans le dataset ?

In [28]:
results_bad_target = run_pipeline_auto(url_tips, "colonne_inexistante", thread_id="bad_target")
print_summary(results_bad_target)

17:54:05 - src.tracking.wandb_tracker - INFO - W&B initialized: project=preprocessing-pipeline, run=tips
17:54:05 - src.agents.base_agent - INFO - Dataset loaded: 244 rows x 7 columns, target='colonne_inexistante'



Dataset: https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv
Target: colonne_inexistante
Statut: ECHEC | Duree: 2.9s
  ERREUR: ValueError: Target column 'colonne_inexistante' not found. Available: ['total_bill', 'tip', 'sex', 'smoker', 'day', 'time', 'size']


## Tableau comparatif des resultats

In [29]:
all_results = {
    "Titanic": results_titanic,
    "Tips": results_tips,
    "Diamonds": results_diamonds,
    "Penguins": results_penguins,
    "Extreme": results_extreme,
    "Bad Target": results_bad_target,
}

summary_rows = []
for name, r in all_results.items():
    n_transforms = len(r.get("transform_proposals", {}).get("transformations", []))
    n_outliers = len(r.get("outlier_proposals", {}).get("outlier_actions", []))
    qs = r.get("quality_score", {})
    ba = r.get("report", {}).get("before_after", {})
    conf_map = r.get("confidence_map", [])
    rule_ct = sum(1 for c in conf_map if c.get("source") == "rule") if conf_map else 0
    total_ct = len(conf_map) if conf_map else 0

    summary_rows.append({
        "Dataset": name,
        "Succes": r["success"],
        "Duree (s)": r["duration"],
        "Domaine": r.get("domain_context", {}).get("domain", "-"),
        "Nb transforms": n_transforms if r["success"] else "-",
        "Nb outlier actions": n_outliers if r["success"] else "-",
        "Quality score": qs.get("overall", "-") if r["success"] else "-",
        "Deterministic %": f"{rule_ct/total_ct*100:.0f}%" if total_ct > 0 else "-",
        "Shape avant": ba.get("original_shape", "-"),
        "Shape apres": ba.get("final_shape", "-"),
        "Erreurs": "; ".join(r["errors"]) if r["errors"] else "",
    })

df_summary = pd.DataFrame(summary_rows)
df_summary

,Dataset,Succes,Duree (s),Domaine,Nb transforms,Nb outlier actions,Quality score,Deterministic %,Shape avant,Shape apres,Erreurs
0,Titanic,True,129.0,transport,19,8,85.8,58%,891 rows x 15 columns,"(712, 20)",
1,Tips,True,108.0,restaurant,9,5,96.7,54%,-,-,
2,Diamonds,True,110.8,e-commerce,23,6,99.8,38%,53940 rows x 10 columns,43152 rows x 30 columns,
3,Penguins,False,32.7,-,-,-,-,-,-,-,ValueError: LLM response failed validation aft...
4,Extreme,True,89.7,autre,11,5,89.0,67%,500 rows x 9 columns,"(397, 408)",
5,Bad Target,False,2.9,-,-,-,-,-,-,-,ValueError: Target column 'colonne_inexistante...


## Analyse des limites

Rappel des limites corrigees depuis la version initiale :

| Limite initiale | Statut | Correction |
|-----------------|--------|------------|
| Pas de validation target | CORRIGE | `_validate_target()` dans base_agent, fail-fast ValueError |
| Colonnes 100% NaN envoyees au LLM | CORRIGE | `_detect_trivial_columns()` + auto-drop deterministe |
| Colonnes constantes ignorees | CORRIGE | Detection auto dans `_detect_trivial_columns()` |
| Haute cardinalite -> OHE explosif | CORRIGE | `frequency_encoding` pour >50 uniques |
| LLM invente des actions | CORRIGE | Pydantic Literal + `validate_llm_response` avec retry |
| Actions inconnues silencieusement skipees | CORRIGE | `raise ValueError` |
| Scalers non persistes | CORRIGE | `joblib.dump` dans `data/outputs/artifacts/` |
| `describe()` trop volumineux | CORRIGE | `_build_col_summary()` compact |

### Limites restantes a verifier

In [30]:
print("VERIFICATION DES LIMITES\n")

# 1. Target invalide -> doit echouer proprement
print("1. Target invalide :")
if not results_bad_target["success"]:
    print(f"   OK: fail-fast avec erreur: {results_bad_target['errors'][0][:100]}")
else:
    print("   REGRESSION: le pipeline accepte une target inexistante")

# 2. Colonnes triviales (100% NaN, constantes, IDs)
print("\n2. Colonnes triviales (extreme dataset) :")
if results_extreme["success"]:
    transforms = results_extreme.get("transform_proposals", {}).get("transformations", [])
    for expected_col, expected_reason in [
        ("all_nan_col", "100% NaN"),
        ("constant_col", "constante"),
        ("high_cardinality_id", "ID unique"),
    ]:
        match = [t for t in transforms if t["column"] == expected_col and t["action"] == "drop_column"]
        status = "OK" if match else "MANQUE"
        print(f"   {status}: {expected_col} ({expected_reason})")

# 3. Datetime
print("\n3. Colonne datetime :")
if results_extreme["success"]:
    dt_actions = [t for t in transforms if t["column"] == "datetime_col"]
    if dt_actions:
        print(f"   Action: {dt_actions[0]['action']} (source: {dt_actions[0].get('source', '?')})")
    else:
        print("   Non traitee (colonne datetime ignoree)")

# 4. Mostly NaN (95%)
print("\n4. Colonne 95% NaN (mostly_nan_col) :")
if results_extreme["success"]:
    nan_actions = [t for t in transforms if t["column"] == "mostly_nan_col"]
    if nan_actions:
        print(f"   Action: {nan_actions[0]['action']} (>50% NaN -> drop attendu)")
    else:
        print("   Non traitee")

# 5. Validation Pydantic
print("\n5. Validation Pydantic (aucune action inventee) :")
if results_extreme["success"]:
    valid_actions = {
        "impute_mean", "impute_median", "impute_mode", "drop_column", "drop_rows",
        "one_hot_encoding", "label_encoding", "ordinal_encoding", "frequency_encoding",
        "standard_scaling", "minmax_scaling", "robust_scaling",
        "log_transform", "sqrt_transform", "extract_datetime",
        "replace_sentinel", "cast_numeric",
    }
    all_transforms_all = []
    for name, r in all_results.items():
        if r["success"]:
            for t in r.get("transform_proposals", {}).get("transformations", []):
                all_transforms_all.append((name, t["column"], t["action"]))
    invalid = [(n, c, a) for n, c, a in all_transforms_all if a not in valid_actions]
    if invalid:
        print(f"   REGRESSION: actions invalides detectees: {invalid}")
    else:
        print(f"   OK: {len(all_transforms_all)} actions, toutes valides")

# 6. Performance gros dataset
print("\n6. Performance Diamonds (54k lignes) :")
print(f"   Duree: {results_diamonds['duration']}s")
if results_diamonds["duration"] > 120:
    print("   Lent (>2min)")
else:
    print(f"   OK")

# 7. Target categorique multi-class (Penguins)
print("\n7. Multi-class (Penguins, 3 especes) :")
if results_penguins["success"]:
    meta = results_penguins.get("final_state", {}).get("target_meta", {})
    print(f"   task_type={meta.get('task_type', '?')}, n_classes={meta.get('n_classes', '?')}")
    print(f"   OK" if meta.get("task_type") == "classification" else "   PROBLEME")

# 8. Ratio deterministe global
print("\n8. Ratio deterministe par dataset :")
for name, r in all_results.items():
    if not r["success"]:
        continue
    cm = r.get("confidence_map", [])
    if cm:
        rule_ct = sum(1 for c in cm if c.get("source") == "rule")
        print(f"   {name}: {rule_ct}/{len(cm)} ({rule_ct/len(cm)*100:.0f}%)")
    else:
        print(f"   {name}: pas de confidence_map")

VERIFICATION DES LIMITES

1. Target invalide :
   OK: fail-fast avec erreur: ValueError: Target column 'colonne_inexistante' not found. Available: ['total_bill', 'tip', 'sex', '

2. Colonnes triviales (extreme dataset) :
   OK: all_nan_col (100% NaN)
   OK: constant_col (constante)
   MANQUE: high_cardinality_id (ID unique)

3. Colonne datetime :
   Action: extract_datetime (source: llm)

4. Colonne 95% NaN (mostly_nan_col) :
   Action: drop_column (>50% NaN -> drop attendu)

5. Validation Pydantic (aucune action inventee) :
   OK: 62 actions, toutes valides

6. Performance Diamonds (54k lignes) :
   Duree: 110.8s
   OK

7. Multi-class (Penguins, 3 especes) :

8. Ratio deterministe par dataset :
   Titanic: 15/26 (58%)
   Tips: 7/13 (54%)
   Diamonds: 11/29 (38%)
   Extreme: 10/15 (67%)


## Pistes d'amelioration restantes

| Piste | Priorite | Description |
|-------|----------|-------------|
| Seuils en config | Moyenne | Deplacer les thresholds hard-codes vers `model_config.yaml` |
| Appels LLM async | Moyenne | Paralleliser les appels LLM independants (domain+diagnosis) |
| Cross-validation score | Basse | Evaluer le dataset transforme avec un modele baseline |
| Multi-provider LLM | Basse | Support Anthropic, Mistral, modeles locaux |
| Feature store | Basse | Integration MLflow/W&B pour le suivi des experiences |
| Sampling gros volumes | Basse | Echantillonner avant `col_summary` pour datasets >50k lignes |